# Monitoring Rules → Gold Alerts (`pipelines/04_monitoring_rules.ipynb`)

**Reads:** `workspace.sentinel_gold.transaction_features` (notebook 03)
**Writes:** `workspace.sentinel_gold.alerts` (one row per flagged transaction)

**Flow:** `read transaction_features → evaluate frozen rules & build evidence → save Gold alerts → verify & summarize`

Three frozen rules decide which transactions need an analyst's review. An alert means "needs review", not "fraud".

| Rule | Flags a transaction when |
| --- | --- |
| `RULE_01_CARD_VELOCITY_1H` | the card already made **3 or more** payments in the last hour |
| `RULE_02_UNUSUAL_AMOUNT_14D` | the card has **5 or more** payments in 14 days **and** this amount is **3 times or more** their average |
| `RULE_03_REPEATED_SIMILAR_MERCHANT_1H` | **2 or more** payments at the same merchant in the last hour with an amount within ±10% |

A transaction matched by several rules still produces **one** alert that lists all matching rules and their evidence.
The thresholds were chosen on January 15–28 and are frozen before evaluating January 29 onward. They must not be tuned with fraud labels.

In [ ]:
from decimal import Decimal
from pyspark.sql import functions as F

GOLD_SCHEMA = "workspace.sentinel_gold"
FEATURES_TABLE = f"{GOLD_SCHEMA}.transaction_features"
GOLD_ALERTS_TABLE = f"{GOLD_SCHEMA}.alerts"
ACTIVE_RULESET_VERSION = "ruleset-v1.0-frozen"

# Rule 1: velocity (frozen)
RULE_01_ID = "RULE_01_CARD_VELOCITY_1H"
RULE_01_VERSION = "v1.0-frozen"
RULE_01_WINDOW = "[event_time - 1h, event_time)"
RULE_01_MIN_PRIOR_CARD_TXNS_1H = 3

# Rule 2: unusual amount (frozen)
RULE_02_ID = "RULE_02_UNUSUAL_AMOUNT_14D"
RULE_02_VERSION = "v1.0-frozen"
RULE_02_WINDOW = "[event_time - 14d, event_time)"
RULE_02_MIN_PRIOR_CARD_TXNS_14D = 5
RULE_02_AMOUNT_MULTIPLIER = Decimal("3.00")

# Rule 3: repeated similar payments (the ±10% band is computed in notebook 03, frozen)
RULE_03_ID = "RULE_03_REPEATED_SIMILAR_MERCHANT_1H"
RULE_03_VERSION = "v1.0-frozen"
RULE_03_WINDOW = "[event_time - 1h, event_time)"
RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H = 2

GOLD_ALERTS_COLUMNS = [
    "alert_id", "trans_num", "ruleset_version", "event_time", "cc_num", "merchant", "category",
    "amount", "replay_batch", "source_file", "source_row_index",
    "card_txn_count_1h", "card_txn_count_14d", "card_avg_amount_14d", "unusual_amount_threshold_14d",
    "card_merchant_txn_count_1h", "card_merchant_similar_amount_count_1h",
    "similar_amount_lower_bound", "similar_amount_upper_bound",
    "matched_rule_count", "matched_rule_ids", "matched_rule_versions", "rule_evidence",
]

## How a rule is applied

For each rule:
1. A **match column** (`True`/`False`) compares the features with the thresholds.
2. An **evidence** value (a `struct`: rule id, version, window, observed numbers, threshold text) is built only when the rule matched.
3. The three evidence values go into one array. `F.array_compact` removes the `NULL`s, so the array contains only matching rules.
4. `rule_evidence.rule_id` reads the `rule_id` field of every struct in the array, giving the list of matched rule IDs without needing `F.transform(..., lambda ...)`.

`F.format_string` fills `%s` with column values. A literal percent sign must be written `%%`.

In [ ]:
def add_rule_matches(features_df):
    df = (
        features_df
        .withColumn(
            "unusual_amount_threshold_14d",
            F.round(F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER), 2).cast("decimal(18,2)"),
        )
        # The bounds from notebook 03 are shown rounded to cents as evidence.
        .withColumn("similar_amount_lower_bound", F.round("similar_amount_lower_bound", 2).cast("decimal(18,2)"))
        .withColumn("similar_amount_upper_bound", F.round("similar_amount_upper_bound", 2).cast("decimal(18,2)"))
        .withColumn("rule_01_velocity_match", F.col("card_txn_count_1h") >= RULE_01_MIN_PRIOR_CARD_TXNS_1H)
        .withColumn(
            "rule_02_unusual_amount_match",
            (F.col("card_txn_count_14d") >= RULE_02_MIN_PRIOR_CARD_TXNS_14D)
            & F.col("card_avg_amount_14d").isNotNull()
            & (F.col("amount") >= F.col("card_avg_amount_14d") * F.lit(RULE_02_AMOUNT_MULTIPLIER)),
        )
        .withColumn(
            "rule_03_similar_merchant_match",
            F.col("card_merchant_similar_amount_count_1h") >= RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H,
        )
    )

    rule_01_evidence = F.when(F.col("rule_01_velocity_match"), F.struct(
        F.lit(RULE_01_ID).alias("rule_id"),
        F.lit(RULE_01_VERSION).alias("rule_version"),
        F.lit(RULE_01_WINDOW).alias("window"),
        F.format_string("card_txn_count_1h=%s", F.col("card_txn_count_1h")).alias("observed_metrics"),
        F.lit(f"card_txn_count_1h >= {RULE_01_MIN_PRIOR_CARD_TXNS_1H}").alias("threshold_definition"),
    ))

    rule_02_evidence = F.when(F.col("rule_02_unusual_amount_match"), F.struct(
        F.lit(RULE_02_ID).alias("rule_id"),
        F.lit(RULE_02_VERSION).alias("rule_version"),
        F.lit(RULE_02_WINDOW).alias("window"),
        F.format_string(
            "amount=%s, card_avg_amount_14d=%s, card_txn_count_14d=%s",
            F.col("amount"), F.col("card_avg_amount_14d"), F.col("card_txn_count_14d"),
        ).alias("observed_metrics"),
        F.format_string(
            f"card_txn_count_14d >= {RULE_02_MIN_PRIOR_CARD_TXNS_14D} AND amount >= {RULE_02_AMOUNT_MULTIPLIER} * card_avg_amount_14d (%s)",
            F.col("unusual_amount_threshold_14d"),
        ).alias("threshold_definition"),
    ))

    rule_03_evidence = F.when(F.col("rule_03_similar_merchant_match"), F.struct(
        F.lit(RULE_03_ID).alias("rule_id"),
        F.lit(RULE_03_VERSION).alias("rule_version"),
        F.lit(RULE_03_WINDOW).alias("window"),
        F.format_string(
            "amount=%s, card_merchant_similar_amount_count_1h=%s, card_merchant_txn_count_1h=%s",
            F.col("amount"), F.col("card_merchant_similar_amount_count_1h"), F.col("card_merchant_txn_count_1h"),
        ).alias("observed_metrics"),
        F.format_string(
            f"card_merchant_similar_amount_count_1h >= {RULE_03_MIN_SIMILAR_MERCHANT_TXNS_1H} within +/-10%% [%s, %s]",
            F.col("similar_amount_lower_bound"), F.col("similar_amount_upper_bound"),
        ).alias("threshold_definition"),
    ))

    return (
        df
        .withColumn("ruleset_version", F.lit(ACTIVE_RULESET_VERSION))
        .withColumn("rule_evidence", F.array_compact(F.array(rule_01_evidence, rule_02_evidence, rule_03_evidence)))
        .withColumn("matched_rule_ids", F.col("rule_evidence.rule_id"))
        .withColumn("matched_rule_versions", F.col("rule_evidence.rule_version"))
        .withColumn("matched_rule_count", F.size("matched_rule_ids"))
        .withColumn("is_flagged", F.col("matched_rule_count") > 0)
    )

## Self-test: just below, exactly at, and above each threshold

Each hand-made row carries the feature values read by `add_rule_matches`.

In [ ]:
rule_fixture_schema = (
    "trans_num string, amount decimal(18,2), card_txn_count_1h long, card_txn_count_14d long, "
    "card_avg_amount_14d decimal(18,2), card_merchant_txn_count_1h long, "
    "card_merchant_similar_amount_count_1h long, "
    "similar_amount_lower_bound decimal(21,4), similar_amount_upper_bound decimal(21,4)"
)


def make_feature_row(trans_num, **changes):
    row = dict(
        trans_num=trans_num, amount=Decimal("10.00"), card_txn_count_1h=0, card_txn_count_14d=0,
        card_avg_amount_14d=None, card_merchant_txn_count_1h=0, card_merchant_similar_amount_count_1h=0,
        similar_amount_lower_bound=Decimal("9.0000"), similar_amount_upper_bound=Decimal("11.0000"),
    )
    row.update(changes)
    return row


rule_fixture_rows = [
    make_feature_row("quiet"),                                                                  # nothing matches
    make_feature_row("velocity_below", card_txn_count_1h=2),                                    # 2 < 3
    make_feature_row("velocity_at", card_txn_count_1h=3),                                       # 3 >= 3
    make_feature_row("velocity_above", card_txn_count_1h=5),                                    # 5 > 3
    make_feature_row("amount_short_history", card_txn_count_14d=4, card_avg_amount_14d=Decimal("10.00"), amount=Decimal("100.00")),
    make_feature_row("amount_below", card_txn_count_14d=5, card_avg_amount_14d=Decimal("10.00"), amount=Decimal("29.99")),
    make_feature_row("amount_at", card_txn_count_14d=5, card_avg_amount_14d=Decimal("10.00"), amount=Decimal("30.00")),
    make_feature_row("amount_above", card_txn_count_14d=6, card_avg_amount_14d=Decimal("10.00"), amount=Decimal("45.00")),
    make_feature_row("similar_below", card_merchant_similar_amount_count_1h=1),                 # 1 < 2
    make_feature_row("similar_at", card_merchant_similar_amount_count_1h=2),                    # 2 >= 2
    make_feature_row("similar_above", card_merchant_similar_amount_count_1h=3),                 # 3 > 2
    make_feature_row("two_rules", card_txn_count_1h=4, card_txn_count_14d=6, card_avg_amount_14d=Decimal("10.00"), amount=Decimal("45.00")),
]

rule_result = add_rule_matches(spark.createDataFrame(rule_fixture_rows, schema=rule_fixture_schema))
actual_rules = {r["trans_num"]: sorted(r["matched_rule_ids"]) for r in rule_result.collect()}

expected_rules = {
    "quiet": [],
    "velocity_below": [],
    "velocity_at": [RULE_01_ID],
    "velocity_above": [RULE_01_ID],
    "amount_short_history": [],
    "amount_below": [],
    "amount_at": [RULE_02_ID],
    "amount_above": [RULE_02_ID],
    "similar_below": [],
    "similar_at": [RULE_03_ID],
    "similar_above": [RULE_03_ID],
    "two_rules": sorted([RULE_01_ID, RULE_02_ID]),
}
assert actual_rules == expected_rules, f"Rule self-test failed.\nExpected: {expected_rules}\nActual:   {actual_rules}"

print("Self-test passed: every threshold behaves correctly just below, at, and above the limit.")

## Apply the rules to all transactions

In [ ]:
features = spark.read.table(FEATURES_TABLE)
assert "is_fraud" not in features.columns, "Fraud labels must not appear in the features"

evaluated = add_rule_matches(features)

rule_counts = evaluated.agg(
    F.count("*").alias("transactions"),
    F.count_if("rule_01_velocity_match").alias("rule_01_matches"),
    F.count_if("rule_02_unusual_amount_match").alias("rule_02_matches"),
    F.count_if("rule_03_similar_merchant_match").alias("rule_03_matches"),
    F.count_if("is_flagged").alias("flagged"),
).collect()[0]
print(rule_counts.asDict())

## Save the alerts

`alert_id` is a deterministic SHA-256 hash of `ruleset_version | trans_num` that stably **identifies** each alert across reruns.
Rebuilding the table in full-snapshot `overwrite` mode from unique `trans_num` feature rows—and verifying `alert_id` and `trans_num` uniqueness below—is what prevents duplicate output rows.

In [ ]:
alerts = (
    evaluated.filter("is_flagged")
    .withColumn("alert_id", F.sha2(F.concat_ws("|", "ruleset_version", "trans_num"), 256))
    .select(*GOLD_ALERTS_COLUMNS)
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
alerts.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_ALERTS_TABLE)

print(f"Saved {rule_counts['flagged']:,} alerts to {GOLD_ALERTS_TABLE}.")

## Verify the saved alerts

In [ ]:
saved_alerts = spark.read.table(GOLD_ALERTS_TABLE)
saved_rows = saved_alerts.count()

assert saved_rows == rule_counts["flagged"], "Saved alerts do not match the flagged count"
assert saved_alerts.select("alert_id").distinct().count() == saved_rows, "Duplicate alert_id"
assert saved_alerts.select("trans_num").distinct().count() == saved_rows, "A transaction has more than one alert"
assert saved_alerts.filter(F.size("rule_evidence") < 1).count() == 0, "An alert has no rule evidence"
assert "is_fraud" not in saved_alerts.columns, "Fraud labels must not appear in alerts"

print(f"Verified {saved_rows:,} alerts: unique IDs, one per transaction, every alert has evidence.")
display(saved_alerts.orderBy("event_time", "trans_num").limit(50))

## Rule-development summary (January 15–28, after 14 days of history)

In [ ]:
development_period = evaluated.filter(F.col("replay_batch").between("2019-01-15", "2019-01-28"))

display(
    development_period.agg(
        F.count("*").alias("evaluated_transactions"),
        F.count_if("is_flagged").alias("flagged_transactions"),
        F.round(F.avg(F.col("is_flagged").cast("int")) * 100, 2).alias("flagged_pct"),
        F.count_if("rule_01_velocity_match").alias("rule_01_velocity_1h_matches"),
        F.count_if("rule_02_unusual_amount_match").alias("rule_02_unusual_amount_14d_matches"),
        F.count_if("rule_03_similar_merchant_match").alias("rule_03_repeated_similar_merchant_1h_matches"),
        F.count_if(F.col("matched_rule_count") > 1).alias("multi_rule_transactions"),
    )
)